# PDA Pipeline · Phase 0: Understand the data

Answers the three sub-tasks of the Notion task *PDA Pipeline · Phase 0: Understand the data*:

| Notion sub-task | Notebook section |
|---|---|
| Inventory + profile every raw file | **1** Inventory, **2** Column profile |
| Classify file roles + test join keys | **3** Keys and roles, **4** Key overlap, **5** Spine and visits |
| Write data dictionary + open questions | **6** Old vs new, **7** Outcomes and mappings, **8** Dictionary and open questions |

**Runs on Bianca: Top Baldy runs it.** The notebook only reads the raw data. It writes reports to `OUT_DIR`, and refuses to write inside `DATA_DIR`.

**Before running:** fill in the three placeholder paths in the config cell. Then run top to bottom. Every section prints what it found, then asks for a decision where the heuristics may be wrong (key column, spine file, time column). Put those decisions into the override variables in the config cell and rerun from there.

**Export rule:** the profile contains min/max and top categories. Treat all outputs as sensitive and review them before anything leaves the cluster.

## 0. Config (fill in the placeholders)

In [ ]:
import re, time, json
from pathlib import Path
import polars as pl

# ---- PATHS: replace the placeholders ---------------------------------------
DATA_DIR     = Path("<PATH_TO_RAW_DATA_DIR>")       # new raw files (read-only)
OUT_DIR      = Path("<PATH_TO_OUTPUT_DIR>")         # reports go here; must be OUTSIDE DATA_DIR
OLD_INFO_DIR = Path("<PATH_TO_OLD_data_info_DIR>")  # old docs: swed_profile.txt + mapping CSVs (optional)

OLD_PROFILE_TXT = OLD_INFO_DIR / "swed_profile.txt"
OLD_LAB_MAP_CSV = OLD_INFO_DIR / "swed_lab_meas_to_snomed.csv"
OLD_UNIT_CSV    = OLD_INFO_DIR / "lab_unit_normalization_07sep2026.csv"

# ---- SETTINGS --------------------------------------------------------------
SUPPORTED_SUFFIXES = {".parquet", ".csv", ".tsv"}
PROFILE_SAMPLE_ROWS = None      # e.g. 1_000_000 for a quick first pass; None = full files
QUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]
CAT_MAX_DISTINCT = 50           # columns with <= this many distinct values get a top-categories list
TOP_N = 10
RERUN_PROFILE = False           # False = reuse per-file profiles already in OUT_DIR/profile
MORTALITY_WINDOW_DAYS = 30      # window used for the mortality prevalence check

# ---- DECISIONS (fill in after reading the output; leave empty at first) ----
KEY_COLUMN_OVERRIDES  = {}      # {"file_stem": "patient_key_column"}
ROLE_OVERRIDES        = {}      # {"file_stem": "spine" | "measurement" | "diagnosis" | "death" | "demographics" | "other"}
SPINE_FILE            = None    # file stem of the visit table, e.g. "my_ed_file"
SPINE_TIME_COL        = None    # arrival timestamp column in the spine file
SPINE_VISIT_ID_COL    = None    # visit id column, if the spine file has one
DEATH_FILE            = None    # file stem of the death table
DEATH_DATE_COL        = None    # date-of-death column in that file
TERM_COLUMN_OVERRIDES = {}      # {"file_stem": "term_name_column"}
UNIT_COLUMN_OVERRIDES = {}      # {"file_stem": "unit_column"}

# ---- NAME HINTS used by the heuristics (lower-case substrings) -------------
KEY_HINTS   = ["lopnr", "patient", "pnr", "person", "pid"]
VISIT_HINTS = ["visit", "besok", "besök", "episode", "encounter", "vtf", "case_id"]
TIME_HINTS  = ["date", "time", "datum", "tid", "dttm", "arrival", "ankomst"]
DUR_HINTS   = ["discharge", "utskriv", "duration", "length", "minutes", "total_min", "los"]
TERM_HINTS  = ["term", "analys", "parameter", "test", "measure"]
VALUE_HINTS = ["value", "varde", "värde", "result", "resultat"]
UNIT_HINTS  = ["unit", "enhet"]
DIAG_HINTS  = ["diag", "icd"]
DEATH_HINTS = ["death", "dod", "död", "mort", "deceased"]
DEMO_HINTS  = ["birth", "fodd", "född", "sex", "kon", "kön", "gender", "age"]

# ---- SAFETY ----------------------------------------------------------------
for name, p in [("DATA_DIR", DATA_DIR), ("OUT_DIR", OUT_DIR)]:
    if "<" in str(p):
        raise RuntimeError(f"{name} is still a placeholder: {p}")
OLD_INFO_AVAILABLE = "<" not in str(OLD_INFO_DIR) and OLD_INFO_DIR.exists()
if not OLD_INFO_AVAILABLE:
    print("NOTE: OLD_INFO_DIR not set or missing; sections 6 and 7c (old-vs-new, mappings) will be skipped.")
DATA_DIR, OUT_DIR = DATA_DIR.resolve(), OUT_DIR.resolve()
if OUT_DIR == DATA_DIR or DATA_DIR in OUT_DIR.parents:
    raise RuntimeError("OUT_DIR must be outside DATA_DIR (raw data is read-only).")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "profile").mkdir(exist_ok=True)

# ---- HELPERS ---------------------------------------------------------------
def scan(path):
    suf = Path(path).suffix.lower()
    if suf == ".parquet":
        return pl.scan_parquet(path)
    if suf in {".csv", ".tsv"}:
        return pl.scan_csv(path, separator="\t" if suf == ".tsv" else ",", infer_schema_length=10000)
    raise ValueError(f"unsupported format: {path}")

def collect(lf):
    return lf.collect(engine="streaming")

def hit(name, hints):
    n = name.lower()
    return any(h in n for h in hints)

def cols_matching(cols, hints):
    return [c for c in cols if hit(c, hints)]

def kind_of(dt):
    if dt.is_float(): return "float"
    if dt.is_integer(): return "int"
    if dt == pl.String: return "string"
    if dt.is_temporal(): return "temporal"
    if dt == pl.Boolean: return "bool"
    return str(dt)

def save_csv(df, name):
    path = OUT_DIR / name
    if DATA_DIR in path.resolve().parents:
        raise RuntimeError("refusing to write inside DATA_DIR")
    df.write_csv(path)
    print(f"  wrote {path}")
    return path

def to_date_expr(c, dt):
    # turn a column into a Date, whatever it is stored as
    if dt == pl.Date: return pl.col(c)
    if dt.is_temporal(): return pl.col(c).cast(pl.Date)
    if dt.is_integer() or dt.is_float():
        return pl.col(c).cast(pl.Int64).cast(pl.String).str.to_date("%Y%m%d", strict=False)
    return pl.col(c).cast(pl.String).str.slice(0, 10).str.to_date(strict=False)

print("config ok; data:", DATA_DIR, "| out:", OUT_DIR)

## 1. Inventory
*Question: for every file, what are the format, size, row count, columns and types, and which column is the patient key?*

In [ ]:
files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES)
if not files:
    raise RuntimeError(f"no supported files under {DATA_DIR}")
paths, schemas, inv_rows, col_rows, errors = {}, {}, [], [], []

for p in files:
    stem = p.stem
    if stem in paths:                                   # same name in two folders
        stem = str(p.relative_to(DATA_DIR)).replace("/", "__").rsplit(".", 1)[0]
    try:
        lf = scan(p)
        schema = lf.collect_schema()
        n_rows = collect(lf.select(pl.len())).item()
    except Exception as e:
        errors.append((str(p), repr(e)))
        print("  FAILED", p.name, repr(e)[:150])
        continue
    paths[stem], schemas[stem] = p, schema
    inv_rows.append(dict(file=stem, relpath=str(p.relative_to(DATA_DIR)), format=p.suffix.lstrip("."),
                         size_mb=round(p.stat().st_size / 1e6, 2), n_rows=n_rows, n_cols=len(schema)))
    for c, dt in schema.items():
        col_rows.append(dict(file=stem, column=c, dtype=str(dt)))

inventory = pl.DataFrame(inv_rows)
columns_long = pl.DataFrame(col_rows)
print(f"{len(inventory)} files, {inventory['size_mb'].sum()/1000:.1f} GB, {inventory['n_rows'].sum():,} rows; {len(errors)} unreadable")
if errors:
    print("UNREADABLE (ask the data provider):", errors)
save_csv(inventory, "inventory.csv")
save_csv(columns_long, "columns.csv")
inventory

## 2. Column profile
*Question: per column, what are the null rate, distinct count, min/max, quantiles and top categories?*
Profiles are cached per file in `OUT_DIR/profile/`, so a rerun after a crash skips finished files.

In [ ]:
def profile_file(stem):
    lf = scan(paths[stem])
    if PROFILE_SAMPLE_ROWS:
        lf = lf.head(PROFILE_SAMPLE_ROWS)
    schema = lf.collect_schema()
    n_rows = collect(lf.select(pl.len())).item()
    exprs = []
    for i, (c, dt) in enumerate(schema.items()):
        col = pl.col(c)
        exprs += [col.null_count().alias(f"{i}__nulls"), col.approx_n_unique().alias(f"{i}__distinct")]
        if dt.is_numeric() or dt.is_temporal() or dt == pl.String:
            exprs += [col.min().alias(f"{i}__min"), col.max().alias(f"{i}__max")]
        if dt.is_numeric():
            exprs += [col.quantile(q).alias(f"{i}__q{int(q*100):02d}") for q in QUANTILES]
    res = collect(lf.select(exprs)).row(0, named=True)
    out = []
    for i, (c, dt) in enumerate(schema.items()):
        distinct = res[f"{i}__distinct"]
        row = dict(file=stem, column=c, dtype=str(dt), n_rows=n_rows,
                   null_rate=round(res[f"{i}__nulls"] / n_rows, 4) if n_rows else None,
                   approx_distinct=distinct,
                   min=str(res[f"{i}__min"]) if f"{i}__min" in res else None,
                   max=str(res[f"{i}__max"]) if f"{i}__max" in res else None)
        for q in QUANTILES:
            k = f"{i}__q{int(q*100):02d}"
            row[f"q{int(q*100):02d}"] = res.get(k)
        row["top_values"] = None
        if distinct is not None and distinct <= CAT_MAX_DISTINCT and dt != pl.Float64:
            top = collect(lf.select(c).group_by(c).len().sort("len", descending=True).head(TOP_N))
            row["top_values"] = "; ".join(f"{r[c]}: {r['len']}" for r in top.iter_rows(named=True))
        out.append(row)
    return pl.DataFrame(out, infer_schema_length=None)

profiles = []
for stem in paths:
    cache = OUT_DIR / "profile" / f"{stem}.csv"
    t0 = time.time()
    try:
        if cache.exists() and not RERUN_PROFILE:
            df = pl.read_csv(cache, infer_schema_length=0)       # all strings, fine for reporting
            print(f"  cached  {stem}")
        else:
            df = profile_file(stem)
            df.write_csv(cache)
            print(f"  profiled {stem} in {time.time()-t0:.0f}s")
        profiles.append(df.with_columns(pl.all().cast(pl.String)))
    except Exception as e:
        errors.append((stem, repr(e)))
        print("  FAILED", stem, repr(e)[:200])
profile_df = pl.concat(profiles, how="diagonal_relaxed")
save_csv(profile_df, "profile_all.csv")
profile_df.select("file", "column", "dtype", "null_rate", "approx_distinct", "min", "max").head(40)

## 3. Keys and roles
*Question: which column is the patient key in each file, which role does each file play (spine, measurements, outcome/diagnosis, demographics), and is the key type consistent?*

Roles are **heuristics from column names**. Check the table, then put corrections in `KEY_COLUMN_OVERRIDES` and `ROLE_OVERRIDES` and rerun this section.

In [ ]:
def pick_key(stem, cols):
    if stem in KEY_COLUMN_OVERRIDES:
        return KEY_COLUMN_OVERRIDES[stem], [KEY_COLUMN_OVERRIDES[stem]]
    cands = cols_matching(cols, KEY_HINTS)
    cands.sort(key=lambda c: (0 if c.lower() == "lopnr_swed" else 1 if "lopnr" in c.lower() else 2, len(c)))
    return (cands[0] if cands else None), cands

def key_stats(stem, key):
    lf = scan(paths[stem])
    dt = schemas[stem][key]
    kind = kind_of(dt)
    notes = []
    base = collect(lf.select(pl.len().alias("n"), pl.col(key).null_count().alias("nulls"))).row(0, named=True)
    if kind == "float":
        nonint = collect(lf.select((pl.col(key) != pl.col(key).floor()).sum())).item()
        if nonint: notes.append(f"{nonint} non-integer float keys")
        expr, canon = pl.col(key).cast(pl.Int64, strict=False), "int"
    elif kind == "int":
        expr, canon = pl.col(key).cast(pl.Int64), "int"
    elif kind == "string":
        stripped = pl.col(key).str.strip_chars()
        bad = collect(lf.select(stripped.cast(pl.Int64, strict=False).null_count() - pl.col(key).null_count())).item()
        if bad == 0:
            expr, canon = stripped.cast(pl.Int64, strict=False), "int"
            notes.append("string key but all numeric")
        else:
            expr, canon = stripped, "string"
            notes.append(f"{bad} non-numeric string keys")
    else:
        expr, canon = pl.col(key).cast(pl.String), "string"
        notes.append(f"unusual key dtype {dt}")
    uniq = collect(lf.select(expr.alias("k")).drop_nulls().unique())["k"]
    n_nonnull = base["n"] - base["nulls"]
    return (dict(file=stem, key=key, key_dtype=str(dt), canon=canon, n_rows=base["n"],
                 key_null_rate=round(base["nulls"] / base["n"], 4) if base["n"] else None,
                 n_unique_keys=len(uniq),
                 rows_per_key=round(n_nonnull / len(uniq), 2) if len(uniq) else None,
                 key_unique_per_row=(len(uniq) == base["n"]),
                 notes="; ".join(notes)), uniq)

feat, key_sets, key_rows = {}, {}, []
for stem, schema in schemas.items():
    cols = list(schema.names())
    key, cands = pick_key(stem, cols)
    f = dict(file=stem, key=key, key_candidates=", ".join(cands),
             visit_cols=cols_matching(cols, VISIT_HINTS),
             time_cols=[c for c in cols if schema[c].is_temporal() or hit(c, TIME_HINTS)],
             dur_cols=cols_matching(cols, DUR_HINTS),
             term_cols=cols_matching(cols, TERM_HINTS), value_cols=cols_matching(cols, VALUE_HINTS),
             unit_cols=cols_matching(cols, UNIT_HINTS), diag_cols=cols_matching(cols, DIAG_HINTS),
             death_cols=cols_matching(cols, DEATH_HINTS), demo_cols=cols_matching(cols, DEMO_HINTS))
    f["rows_per_key"] = None
    if key:
        try:
            ks, uniq = key_stats(stem, key)
            key_rows.append(ks); key_sets[stem] = (ks["canon"], uniq)
            f["rows_per_key"] = ks["rows_per_key"]
        except Exception as e:
            errors.append((stem, repr(e))); print("  key test FAILED", stem, repr(e)[:150])
    feat[stem] = f

def guess_role(f):
    stem, rpk = f["file"], f["rows_per_key"]
    if hit(stem, DEATH_HINTS) or (f["death_cols"] and not f["term_cols"]): return "death"
    if f["term_cols"] and f["value_cols"] and rpk and rpk > 2: return "measurement"
    if f["diag_cols"]: return "diagnosis"
    if rpk is not None and rpk <= 1.05 and f["demo_cols"]: return "demographics"
    if f["time_cols"] and f["key"]: return "spine"
    if f["demo_cols"]: return "demographics"
    return "other"

for stem, f in feat.items():
    f["role_guess"] = guess_role(f)
    f["role"] = ROLE_OVERRIDES.get(stem, f["role_guess"])

keys_df = pl.DataFrame(key_rows)
roles_df = pl.DataFrame([dict(file=s, role=f["role"], role_guess=f["role_guess"], key=f["key"],
                              key_candidates=f["key_candidates"], rows_per_key=f["rows_per_key"],
                              time_cols=", ".join(f["time_cols"]), visit_cols=", ".join(f["visit_cols"]),
                              term_cols=", ".join(f["term_cols"]), diag_cols=", ".join(f["diag_cols"]),
                              death_cols=", ".join(f["death_cols"])) for s, f in feat.items()])
save_csv(keys_df, "key_tests.csv")
save_csv(roles_df, "file_roles.csv")

print("\nFiles without any key candidate (check by hand):", [s for s, f in feat.items() if not f["key"]])
print("Key kinds after normalising:", keys_df.group_by("canon").len().to_dicts())
print("Raw key dtypes in use:", keys_df.group_by("key_dtype").len().to_dicts())
roles_df

In [ ]:
# Key problems at a glance
keys_df.filter(
    (pl.col("key_null_rate") > 0) | (pl.col("notes") != "")
).select("file", "key", "key_dtype", "key_null_rate", "notes")

## 4. Key overlap between files
*Question: which files share patients? Cell (row A, column B) = share of A's patients that also exist in B.*
Low shares from the spine to a measurement file are normal (not everyone has every test); a share near 0 means the key does not match.

In [ ]:
names = list(key_sets)
def as_comparable(stem, other):
    kind, s = key_sets[stem]
    return s if kind == key_sets[other][0] else s.cast(pl.String)

ov_rows = []
for a in names:
    for b in names:
        if a == b:
            continue
        sa, sb = as_comparable(a, b), as_comparable(b, a)
        ov_rows.append(dict(a=a, b=b, share_of_a_in_b=round(float(sa.is_in(sb).sum()) / max(len(sa), 1), 4)))
overlap_long = pl.DataFrame(ov_rows)
overlap_matrix = overlap_long.pivot(on="b", index="a", values="share_of_a_in_b").sort("a")
save_csv(overlap_long, "key_overlap_long.csv")
save_csv(overlap_matrix, "key_overlap_matrix.csv")
print("Pairs with near-zero overlap (possible key mismatch):")
print(overlap_long.filter(pl.col("share_of_a_in_b") < 0.01).sort("a").head(30))
overlap_matrix

## 5. Spine and visits
*Question: which file is one row per visit, does visit time exist, and what does the revisit structure look like?*

In [ ]:
spine_cands = [f for f in feat.values() if f["role"] == "spine" and f["key"]]
print("Spine candidates (role == spine):")
for f in spine_cands:
    ks = keys_df.filter(pl.col("file") == f["file"]).row(0, named=True)
    print(f"  {f['file']:40s} rows={ks['n_rows']:>12,}  patients={ks['n_unique_keys']:>10,}  rows/patient={ks['rows_per_key']}"
          f"  time_cols={f['time_cols'][:3]}  visit_cols={f['visit_cols'][:2]}  duration_cols={f['dur_cols'][:2]}")

spine = SPINE_FILE or (max(spine_cands, key=lambda f: len(f["time_cols"]) + 2 * len(f["visit_cols"]) + len(f["dur_cols"]))["file"] if spine_cands else None)
if spine is None:
    raise RuntimeError("No spine candidate found: set SPINE_FILE (this is an open question for the data provider).")
sf = feat[spine]
SP_KEY = sf["key"]
SP_TIME = SPINE_TIME_COL or (sf["time_cols"][0] if sf["time_cols"] else None)
SP_VISIT = SPINE_VISIT_ID_COL or (sf["visit_cols"][0] if sf["visit_cols"] else None)
print(f"\nUsing spine = {spine} | key = {SP_KEY} | arrival time = {SP_TIME} | visit id = {SP_VISIT}")
print("  (override with SPINE_FILE / SPINE_TIME_COL / SPINE_VISIT_ID_COL if wrong)")
print("  other time columns:", sf["time_cols"], "| duration-like columns:", sf["dur_cols"])

In [ ]:
sp = scan(paths[spine])
sp_schema = schemas[spine]
res = {}
res["rows"] = collect(sp.select(pl.len())).item()
res["patients"] = collect(sp.select(pl.col(SP_KEY).n_unique())).item()
if SP_VISIT:
    res["distinct_visit_ids"] = collect(sp.select(pl.col(SP_VISIT).n_unique())).item()
    res["visit_id_unique_per_row"] = res["distinct_visit_ids"] == res["rows"]
if SP_TIME:
    res["distinct_key_time_pairs"] = collect(sp.select(pl.struct(SP_KEY, SP_TIME).n_unique())).item()
    res["exact_duplicate_key_time_rows"] = res["rows"] - res["distinct_key_time_pairs"]
    tcol = to_date_expr(SP_TIME, sp_schema[SP_TIME])
    span = collect(sp.select(tcol.min().alias("first"), tcol.max().alias("last"), tcol.null_count().alias("unparseable_or_null")))
    res.update(span.row(0, named=True))
    yearly = collect(sp.select(tcol.dt.year().alias("year")).group_by("year").len().sort("year"))
    print("Visits per year:"); print(yearly)
    save_csv(yearly, "spine_visits_per_year.csv")
print(json.dumps({k: str(v) for k, v in res.items()}, indent=2))

per_patient = collect(sp.group_by(SP_KEY).len().select(
    pl.col("len").quantile(0.5).alias("median"), pl.col("len").quantile(0.95).alias("p95"),
    pl.col("len").max().alias("max"), (pl.col("len") > 1).mean().alias("share_with_revisits")))
print("\nVisits per patient:"); print(per_patient)

## 6. Old vs new schemas
*Question: how different is the new data from the old SWED files? Which columns were added, dropped or retyped, and which new files resemble which old files?*

In [ ]:
def parse_old_profile(path):
    # reads "# Profiling .../<name>.parquet" blocks and their "## Parquet schema" sections
    old, cur, in_schema = {}, None, False
    for line in Path(path).read_text(errors="replace").splitlines():
        m = re.match(r"#\s*Profiling\s+(\S+)", line)
        if m:
            cur = Path(m.group(1)).stem; old[cur] = {}; in_schema = False; continue
        if line.startswith("## "):
            in_schema = line.strip() == "## Parquet schema"; continue
        if cur and in_schema and ":" in line and not line.startswith(" "):
            k, v = line.split(":", 1)
            old[cur][k.strip()] = v.strip()
    return {k: v for k, v in old.items() if v}

schema_diff = None
if OLD_INFO_AVAILABLE and OLD_PROFILE_TXT.exists():
    old_schemas = parse_old_profile(OLD_PROFILE_TXT)
    print(f"old profile parsed: {len(old_schemas)} files")
    rows = []
    for stem, schema in schemas.items():
        new_cols = {c: str(dt) for c, dt in schema.items()}
        def jac(o):
            a, b = set(new_cols), set(old_schemas[o]); return len(a & b) / max(len(a | b), 1)
        if stem in old_schemas:
            match, how = stem, "same name"
        else:
            match = max(old_schemas, key=jac); how = f"closest by columns (Jaccard {jac(match):.2f})"
            if jac(match) < 0.3:
                match, how = None, "no similar old file"
        if match:
            o = old_schemas[match]
            rows.append(dict(new_file=stem, old_file=match, match=how,
                             columns_added=", ".join(sorted(set(new_cols) - set(o))),
                             columns_dropped=", ".join(sorted(set(o) - set(new_cols))),
                             type_changed=", ".join(f"{c} ({o[c]} -> {new_cols[c]})" for c in sorted(set(o) & set(new_cols)) if o[c] != new_cols[c])))
        else:
            rows.append(dict(new_file=stem, old_file=None, match=how, columns_added="", columns_dropped="", type_changed=""))
    schema_diff = pl.DataFrame(rows)
    save_csv(schema_diff, "schema_old_vs_new.csv")
else:
    print("skipped: old profile not available")
schema_diff

## 7. Outcomes and mappings
*Questions: where could the **diagnosis** and **mortality** labels come from? How much of the new lab/term vocabulary is covered by the old SNOMED and unit mappings?*

In [ ]:
# --- 7a. Diagnosis files ------------------------------------------------------
diag_summary = []
for stem, f in feat.items():
    if f["role"] != "diagnosis" or not f["key"]:
        continue
    lf = scan(paths[stem])
    for c in f["diag_cols"][:3]:
        if schemas[stem][c] != pl.String:
            continue
        top = collect(lf.select(pl.col(c).str.slice(0, 3).alias("code3")).drop_nulls().group_by("code3").len().sort("len", descending=True).head(15))
        diag_summary.append(dict(file=stem, column=c, distinct_codes=collect(lf.select(pl.col(c).n_unique())).item(),
                                 null_rate=collect(lf.select(pl.col(c).null_count() / pl.len())).item(),
                                 has_time_col=bool(f["time_cols"]), time_cols=", ".join(f["time_cols"][:3]),
                                 top_3char_codes="; ".join(f"{r['code3']}: {r['len']}" for r in top.iter_rows(named=True))))
diag_df = pl.DataFrame(diag_summary) if diag_summary else None
if diag_df is not None:
    save_csv(diag_df, "diagnosis_candidates.csv")
else:
    print("no diagnosis file recognised: open question for the data provider")
diag_df

In [ ]:
# --- 7b. Mortality: how many spine visits have a death record, and when? -----
death = DEATH_FILE or next((s for s, f in feat.items() if f["role"] == "death" and f["key"] and (f["time_cols"] or f["death_cols"])), None)
mort = {}
if death:
    df_ = feat[death]
    dcol = DEATH_DATE_COL or next((c for c in df_["death_cols"] + df_["time_cols"] if kind_of(schemas[death][c]) in {"temporal", "string", "int", "float"}), None)
    print(f"Using death file = {death} | death date column = {dcol}   (override with DEATH_FILE / DEATH_DATE_COL)")
    if dcol and SP_TIME:
        vis = scan(paths[spine]).select(pl.col(SP_KEY).alias("k"), to_date_expr(SP_TIME, schemas[spine][SP_TIME]).alias("arrival"))
        dth = scan(paths[death]).select(pl.col(df_["key"]).alias("k"), to_date_expr(dcol, schemas[death][dcol]).alias("died")).group_by("k").agg(pl.col("died").min())
        j = vis.join(dth, on="k", how="left").with_columns((pl.col("died") - pl.col("arrival")).dt.total_days().alias("days"))
        s = collect(j.select(pl.len().alias("visits"), pl.col("died").is_not_null().sum().alias("with_death_record"),
                             (pl.col("days") < 0).sum().alias("death_before_arrival"),
                             ((pl.col("days") >= 0) & (pl.col("days") <= MORTALITY_WINDOW_DAYS)).sum().alias(f"died_within_{MORTALITY_WINDOW_DAYS}d"),
                             pl.col("arrival").null_count().alias("arrival_unparseable")))
        mort = s.row(0, named=True)
        mort["prevalence_in_window"] = round(mort[f"died_within_{MORTALITY_WINDOW_DAYS}d"] / mort["visits"], 5)
        print(json.dumps(mort, indent=2))
        save_csv(pl.DataFrame([{k: str(v) for k, v in mort.items()}]), "mortality_check.csv")
    else:
        print("Need DEATH_DATE_COL and SPINE_TIME_COL to compute mortality prevalence.")
else:
    print("no death file recognised: open question for the data provider")

In [ ]:
# --- 7c. Measurement terms vs the old SNOMED / unit mappings -----------------
term_cov, unit_issues = [], []
if OLD_INFO_AVAILABLE and OLD_LAB_MAP_CSV.exists():
    old_terms = set(pl.read_csv(OLD_LAB_MAP_CSV, infer_schema_length=0)["term_name"].str.strip_chars().str.to_lowercase().to_list())
    old_units = None
    if OLD_UNIT_CSV.exists():
        u = pl.read_csv(OLD_UNIT_CSV, infer_schema_length=0)
        old_units = set(zip(u["term_name"].str.to_lowercase().to_list(), u["unit"].str.to_lowercase().to_list()))
    for stem, f in feat.items():
        if f["role"] != "measurement":
            continue
        tc = TERM_COLUMN_OVERRIDES.get(stem) or next((c for c in f["term_cols"] if schemas[stem][c] == pl.String), None)
        if tc is None:
            print(f"  {stem}: no string term column found; set TERM_COLUMN_OVERRIDES"); continue
        lf = scan(paths[stem])
        terms = collect(lf.select(pl.col(tc).str.strip_chars().str.to_lowercase().alias("t")).group_by("t").len())
        terms = terms.with_columns(pl.col("t").is_in(list(old_terms)).alias("mapped"))
        n_rows_all, n_rows_mapped = terms["len"].sum(), terms.filter(pl.col("mapped"))["len"].sum()
        term_cov.append(dict(file=stem, term_column=tc, distinct_terms=len(terms), mapped_terms=int(terms["mapped"].sum()),
                             share_rows_mapped=round(n_rows_mapped / max(n_rows_all, 1), 4)))
        top_un = terms.filter(~pl.col("mapped")).sort("len", descending=True).head(30)
        save_csv(top_un.rename({"t": "term", "len": "rows"}).drop("mapped"), f"unmapped_terms_{stem}.csv")
        uc = UNIT_COLUMN_OVERRIDES.get(stem) or next((c for c in f["unit_cols"] if schemas[stem][c] == pl.String), None)
        if uc:
            tu = collect(lf.select(pl.col(tc).str.strip_chars().str.to_lowercase().alias("t"), pl.col(uc).str.strip_chars().str.to_lowercase().alias("u")).group_by("t", "u").len())
            multi = tu.group_by("t").len().filter(pl.col("len") > 1).height
            unseen = sum(1 for t_, u_ in zip(tu["t"].to_list(), tu["u"].to_list()) if (t_, u_) not in old_units) if old_units else None
            unit_issues.append(dict(file=stem, unit_column=uc, terms_with_several_units=multi, term_unit_pairs_not_in_old_table=unseen))
else:
    print("skipped: old mapping CSV not available")
term_cov_df = pl.DataFrame(term_cov) if term_cov else None
if term_cov_df is not None:
    save_csv(term_cov_df, "term_mapping_coverage.csv")
if unit_issues:
    save_csv(pl.DataFrame(unit_issues), "unit_issues.csv")
term_cov_df

## 8. Data dictionary and open questions
*Deliverables of the last sub-task:* `data_dictionary.md` (the equivalent of `swed_profile.txt`) and `open_questions.md` (for whoever delivered the data). Findings are written from what the sections above measured, plus a fixed list of questions the data alone cannot answer.

In [ ]:
L = ["# Data dictionary (new dataset)", "",
     f"Generated {time.strftime('%Y-%m-%d %H:%M')} from {len(paths)} files. Roles are heuristics unless overridden.", "",
     "| file | role | rows | size MB | key | key dtype | patients | rows/patient |", "|---|---|---|---|---|---|---|---|"]
inv_idx = {r["file"]: r for r in inventory.iter_rows(named=True)}
key_idx = {r["file"]: r for r in keys_df.iter_rows(named=True)}
for stem, f in feat.items():
    i, k = inv_idx[stem], key_idx.get(stem, {})
    L.append(f"| {stem} | {f['role']} | {i['n_rows']:,} | {i['size_mb']} | {f['key']} | {k.get('key_dtype','')} | {k.get('n_unique_keys','')} | {k.get('rows_per_key','')} |")
L += ["", f"**Spine:** `{spine}`, key `{SP_KEY}`, arrival time `{SP_TIME}`, visit id `{SP_VISIT}`.", ""]
for stem, f in feat.items():
    L += [f"## {stem}  ({f['role']})", "", "| column | type | null rate | distinct (approx) | min | max | top values |", "|---|---|---|---|---|---|---|"]
    for r in profile_df.filter(pl.col("file") == stem).iter_rows(named=True):
        tv = (r["top_values"] or "")[:120].replace("|", "/")
        L.append(f"| {r['column']} | {r['dtype']} | {r['null_rate']} | {r['approx_distinct']} | {r['min'] or ''} | {r['max'] or ''} | {tv} |")
    L.append("")
dict_path = OUT_DIR / "data_dictionary.md"
dict_path.write_text("\n".join(L))
print("wrote", dict_path)

In [ ]:
Q_auto = []
if errors:
    Q_auto.append(f"{len(errors)} file(s) could not be read or profiled ({', '.join(str(e[0]) for e in errors[:5])}). Are they corrupted or in another format?")
for stem, f in feat.items():
    if not f["key"]:
        Q_auto.append(f"`{stem}` has no recognisable patient key column. Which column links it to patients?")
bad = keys_df.filter((pl.col("key_null_rate") > 0.001) | pl.col("notes").str.contains("non-integer|non-numeric|unusual"))
for r in bad.iter_rows(named=True):
    Q_auto.append(f"`{r['file']}`: key `{r['key']}` ({r['key_dtype']}) has issues: null rate {r['key_null_rate']}, {r['notes'] or 'none'}. Is this the intended key?")
if keys_df["key_dtype"].n_unique() > 1:
    Q_auto.append("The patient key has different types across files (" + ", ".join(f"{d['key_dtype']}: {d['len']} files" for d in keys_df.group_by("key_dtype").len().to_dicts()) + "). Is it the same identifier everywhere? (The old data used a double throughout.)")
low = overlap_long.filter((pl.col("a") == spine) & (pl.col("share_of_a_in_b") < 0.05)).get_column("b").to_list() if spine in key_sets else []
if low:
    Q_auto.append(f"Almost none of the spine patients appear in: {', '.join(low)}. Is the patient key the same there, or are these files for a different population?")
if not SP_TIME:
    Q_auto.append("The spine file has no arrival time column. Is there visit time elsewhere? Without it measurements cannot be linked to visits.")
if not sf["dur_cols"]:
    Q_auto.append("No visit duration / discharge time column found in the spine. The old linking rule (60 min before arrival to arrival + visit duration) needs one; what should the window be?")
if SP_VISIT is None:
    Q_auto.append("No visit id column in the spine. Is (patient, arrival time) a valid visit identifier?")
elif not res.get("visit_id_unique_per_row", True):
    Q_auto.append(f"Visit id `{SP_VISIT}` is not unique per row in `{spine}`. What does one row represent?")
if res.get("exact_duplicate_key_time_rows", 0):
    Q_auto.append(f"{res['exact_duplicate_key_time_rows']:,} spine rows share the same patient and arrival time. Duplicates, or parallel records?")
if diag_df is None:
    Q_auto.append("No diagnosis table was recognised. Where do diagnoses come from, and how are they linked to visits?")
if not death:
    Q_auto.append("No death table was recognised. Where does mortality come from?")
if mort.get("death_before_arrival"):
    Q_auto.append(f"{mort['death_before_arrival']:,} visits have a death date before arrival. Data error, or a different date meaning?")
if term_cov_df is not None:
    for r in term_cov_df.iter_rows(named=True):
        if r["share_rows_mapped"] < 0.8:
            Q_auto.append(f"`{r['file']}`: only {r['share_rows_mapped']:.0%} of rows have terms in the old SNOMED mapping. Is there a new mapping, or do we extend ours?")
for r in unit_issues:
    if r["terms_with_several_units"]:
        Q_auto.append(f"`{r['file']}`: {r['terms_with_several_units']} terms occur with several units. Is there a unit table from the provider?")

Q_fixed = [
    "Which outcome definitions are intended: which diagnosis (codes, or a group), and mortality within how many days of arrival?",
    "Is the new data a different set of patients from the old SWED data, or does it overlap? (Matters for using it as an independent test set.)",
    "Are there test or dummy patients to exclude (the old data had one, `lopnr_swed` 701403)?",
    "Is anything pseudonymised differently between files (keys re-generated per delivery)?",
    "Are timestamps in local time or UTC, and are there time zone or daylight-saving issues?",
    "What does a missing value mean for each measurement (not measured, not recorded, or a lab failure)?",
    "Is there a documentation file or variable list for any of the files, even partial?",
    "Is the file list complete? Are any tables known to be missing?",
]
out = ["# Open questions for the data provider", "",
       "## From the data (generated)", ""] + [f"{i}. {q}" for i, q in enumerate(Q_auto, 1)] + \
      ["", "## Always needed", ""] + [f"{i}. {q}" for i, q in enumerate(Q_fixed, 1)]
(OUT_DIR / "open_questions.md").write_text("\n".join(out))
print("\n".join(out))

## Outputs and next step

Everything is in `OUT_DIR`:

| File | Answers |
|---|---|
| `inventory.csv`, `columns.csv` | files, sizes, rows, columns and types |
| `profile_all.csv`, `profile/` | per-column profile |
| `key_tests.csv`, `file_roles.csv` | key column, type, uniqueness and role per file |
| `key_overlap_matrix.csv`, `key_overlap_long.csv` | patient overlap between files |
| `spine_visits_per_year.csv` | visit counts over time in the spine |
| `schema_old_vs_new.csv` | new vs old schemas |
| `diagnosis_candidates.csv`, `mortality_check.csv` | label sources |
| `term_mapping_coverage.csv`, `unmapped_terms_*.csv`, `unit_issues.csv` | vocabulary and unit coverage |
| `data_dictionary.md`, `open_questions.md` | final deliverables of Phase 0 |

**Next:** send `open_questions.md` to the data provider; put the confirmed key, role and spine decisions into the config cell; then move on to Phase 1 (file manifest and streaming reads).